## Guardrails

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
## Quick illustration of the two approaches

import re

def deterministic_guardrail(text: str) -> bool:
    """ Return True if content is blocked. """
    banned_keywords = ["hack","exploit","malware","bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the captial of france?",
    "Explain how malware spreads"
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}:{inp}")



=== Deterministic Guardrail Demo ===
Blocked:How do I hack into a database?
Allowed:What is the captial of france?
Blocked:Explain how malware spreads


In [4]:
from langchain_groq import ChatGroq

## -- Model-Based approach
def model_based_guardrail(text: str) -> str:
    model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    prompt = f"""You are a strict content safety classifier.
Mark input as UNSAFE if it requests: hacking/unauthorized access, malware creation, 
weapons/explosives, illegal activity instructions, or other harmful content — 
even if phrased as a general question.
Mark input as SAFE only for benign, harmless queries.


Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("--- Model-Based GuardRail Demo ---")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "Unsafe" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}:{inp}")

/Users/shivanshmishra2701gmail.com/Desktop/AgenticWorkspace/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Model-Based GuardRail Demo ---
Unsafe:How do I hack into a database?
SAFE:What is the captial of france?
Unsafe:Explain how malware spreads


## Built in Guardrail -- PII Detection Middleware (personal identifable information)


In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

## Define a simple dummy tool

@tool
def customer_lookup(query: str) -> str:
    """ Look up customer information """
    return f"Customer record found for query:{query}"


agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [customer_lookup],
    middleware = [
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input = True,
        ),
        ## Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy = "mask",
            apply_to_input = True,
        ),
        PIIMiddleware(
            "api_key",
            detector = r"sk-[a-zA-Z0-9]{32}",
            strategy = "block",
            apply_to_input = True,
        ),
    ],
)

print("Agent with PII middleware created successfully")

Agent with PII middleware created successfully


In [6]:
# Test PII Redaction

result = agent.invoke({
    "messages":[{
        "role":"user",
        "content" :"My email is john.doe@example.com and my card is 5105-1051-5100.Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I've located your customer record. Is there something specific you'd like to know or discuss regarding your account?


In [7]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is 5105-1051-5100.Can you help me?', additional_kwargs={}, response_metadata={}, id='90137ab6-38eb-463d-bfb6-84af85dac993'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'y0wx43dqr', 'function': {'arguments': '{"query":"[REDACTED_EMAIL] 5105-1051-5100"}', 'name': 'customer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 236, 'total_tokens': 265, 'completion_time': 0.06393528, 'completion_tokens_details': None, 'prompt_time': 0.011849526, 'prompt_tokens_details': None, 'queue_time': 0.161998642, 'total_time': 0.075784806}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f66c0-1c0d-7380-84f6-09c4b5f84c5c-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query': '[REDACTED_EMAIL]

## Human in the loop middleware 

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool 
def search_web(query: str) -> str:
    """ Search the web for information. """
    return f"Search results for : {query}"

@tool
def send_email(to : str,subject:str,body:str) -> str:
    """ Send an email to a recipient. """
    return f"Email sent to {to} with subjects. "

@tool
def delete_records(table:str,condition:str)->str:
    """Delete records from the database """
    return f"Deleted record from {table} where {condition}"

## Create agent with HITL middlware
hitl_agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [search_web,send_email,delete_records],
    middleware = [
 HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email":True,
                "delete_records": True,
                "search_web":False,
            }
        ),
    ],
    checkpointer = InMemorySaver(),
)

print("Human in the Loop agent created")

Human in the Loop agent created


In [9]:
config = {"configurable": {"thread_id":"session_001"}}

result = hitl_agent.invoke(
    {"messages":[{"role":"user","content":"Send an email to the team@company.com about the Q4 results"}]},
    config = config,
)

print("=== Agent Paused - awaiting human approval ===")
print(result)

=== Agent Paused - awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to the team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='9f97a37e-c2df-4ae4-9814-20e7ce8f7775'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'k12gjk5hb', 'function': {'arguments': '{"body":"Please find the Q4 results attached","subject":"Q4 Results","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 348, 'total_tokens': 386, 'completion_time': 0.095408323, 'completion_tokens_details': None, 'prompt_time': 0.017012924, 'prompt_tokens_details': None, 'queue_time': 0.057940065, 'total_time': 0.112421247}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f66c0-1f09-7561-a6f3-e7df2d611785-0', t

In [10]:
# Human reviews and approves
approved_result = hitl_agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config = config
)

In [11]:
print("=== Approved! final Response ")
print(approved_result["messages"][-1].content)

=== Approved! final Response 



## Custom Guardrails 

In [17]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """ Deterministic guarrail 
    """
    def __init__(self,banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state:AgentState, runtime : Runtime) -> dict [str,Any] | None:
        if not state["messages"]:
            return None
        first_message = state["messages"][0]
        if first_message.type != "human":
            return None
        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked - keyword detected: '{keyword}'")
                return{
                    "messages":[{
                        "role":"assistant",
                        "content":(
                            "I cannot process requests containing inappropriate content."
                            "Please rephrases your request."
                        )
                    }],
                    "jump_to":"end"
                }
            return None
    
@tool 
def search_tool(query:str)->str:
    """Search for information."""
    return f"Result for : {query}"
    
    #Create agent with content filter
filtered_agent = create_agent(
    model="groq:qwen/qwen3-32b",
     tools = [search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack","exploit","malware","jailbreak","bypass"]
            ),
        ],
    )

print("Content filter agent created! ")

Content filter agent created! 


In [19]:
# Test
result = filtered_agent.invoke({
    "messages": [{"role":"user","content":"What is machine learning"}]
})

print("Safe request response")
print(result ["messages"][-1].content)

Safe request response
Machine learning is a subset of artificial intelligence (AI) that enables systems to automatically learn and improve from experience without being explicitly programmed. It involves developing algorithms that analyze data, identify patterns, and make decisions or predictions with minimal human intervention. 

Key concepts include:
- **Supervised learning**: Using labeled data to train models (e.g., predicting house prices based on historical data).
- **Unsupervised learning**: Finding hidden patterns in unlabeled data (e.g., clustering customers by behavior).
- **Reinforcement learning**: Learning optimal actions through trial-and-error interactions (e.g., training a robot to walk).

Common applications include image recognition, spam detection, recommendation systems, and self-driving cars. The process typically involves preparing training data, selecting a model, training it to minimize errors, and deploying it for predictions.


In [21]:
#Test
result = filtered_agent.invoke({
    "messages":[{"role":"user","content":"How do i hack the server"}]
})
print("Unsafe request response.")
print(result["messages"][-1].content)

Blocked - keyword detected: 'hack'
Unsafe request response.
I cannot process requests containing inappropriate content.Please rephrases your request.
